# 🤖 AutoML Pilot - Pro Training Template
Use this Colab notebook to run advanced AutoML (PyCaret) on your dataset with automated reporting, EDA, and SHAP interpretation.

In [ ]:
# @title 1. Install Dependencies
!pip install -q pycaret[full] ydata-profiling pandas jinja2 shap
print('✅ Dependencies installed!')

In [ ]:
# @title 2. Load Dataset
import pandas as pd
import os
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print(f'✅ Loaded {filename}: {df.shape[0]} rows')

In [ ]:
# @title 3. Automated EDA
run_eda = True # @param {type:"boolean"}
if run_eda:
    from ydata_profiling import ProfileReport
    profile = ProfileReport(df, title="Profiling Report", explorative=True)
    profile.to_notebook_iframe()

In [ ]:
# @title 4. AutoML Setup & Training
target_column = 'target' # @param {type:"string"}
task_type = 'classification' # @param ["classification", "regression"]

if task_type == 'classification':
    from pycaret.classification import setup, compare_models, pull, save_model, finalize_model, interpret_model
    s = setup(data=df, target=target_column, session_id=123, verbose=False)
    best_model = compare_models()
    leaderboard = pull()
    final_model = finalize_model(best_model)
    save_model(final_model, 'best_model')
else:
    from pycaret.regression import setup, compare_models, pull, save_model, finalize_model, interpret_model
    s = setup(data=df, target=target_column, session_id=123, verbose=False)
    best_model = compare_models()
    leaderboard = pull()
    final_model = finalize_model(best_model)
    save_model(final_model, 'best_model')

display(leaderboard.head())

In [ ]:
# @title 5. Model Interpretation (SHAP)
try:
    interpret_model(best_model)
except Exception as e:
    print(f'Could not run SHAP: {e}')

In [ ]:
# @title 6. Automated Email Reporting
import smtplib, ssl
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from jinja2 import Template

recipient_email = '' # @param {type:"string"}
sender_email = '' # @param {type:"string"}
sender_password = '' # @param {type:"string"}

if recipient_email and sender_email and sender_password:
    msg = MIMEMultipart("alternative")
    msg["Subject"] = f"🚀 AutoML Pilot Pro - {task_type.capitalize()} Report"
    msg["From"] = sender_email
    msg["To"] = recipient_email

    html_template = """
    <html>
    <body style='font-family: Arial, sans-serif;'>
        <div style='background-color: #f4f4f4; padding: 20px;'>
            <h2 style='color: #2e6c80;'>🚀 AutoML Training Report</h2>
            <p>Your model has been trained successfully on Google Colab.</p>
            <h3>📊 Leaderboard Summary</h3>
            {{ table }}
            <p style='font-size: 0.8em; color: #777;'>Sent via AutoML Pilot Pro Colab Integration</p>
        </div>
    </body>
    </html>
    """
    template = Template(html_template)
    html_content = template.render(table=leaderboard.head().to_html())
    
    msg.attach(MIMEText(html_content, "html"))
    context = ssl.create_default_context()
    with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=context) as server:
        server.login(sender_email, sender_password)
        server.sendmail(sender_email, recipient_email, msg.as_string())
    print('✅ Email report sent!')

In [ ]:
# @title 7. Download Model
from google.colab import files
if os.path.exists('best_model.pkl'):
    files.download('best_model.pkl')
    print('✅ Model downloaded!')